# Section 1: Salesforce Zero-Copy Integration Demo
**ACME IM | Distribution Insights**

Salesforce → Snowflake via Snowpipe V2. No ETL, no data movement.
Advisor signals computed live by joining SFDC opportunities with Snowflake dims.

In [ ]:
import os, tomllib, json, urllib.request, urllib.error
from cryptography.hazmat.primitives.serialization import load_pem_private_key
from snowflake.snowpark import Session
from snowflake.cortex import Complete
import snowflake.connector
import pandas as pd

# Load keypair auth from connections.toml (works locally; get_active_session() used in SiS)
_CONN_NAME = os.environ.get('SNOWFLAKE_DEFAULT_CONNECTION_NAME', 'your_connection')
with open(os.path.expanduser('~/.snowflake/connections.toml'), 'rb') as f:
    _cfg = tomllib.load(f)[_CONN_NAME]

_key_path = os.path.expanduser(_cfg['private_key_path'])
with open(_key_path, 'rb') as f:
    _private_key = load_pem_private_key(f.read(), password=None)

_params = {k: v for k, v in _cfg.items() if k != 'private_key_path'}
_params['private_key'] = _private_key

# Snowpark Session
session = Session.builder.configs(_params).create()
session.sql('USE WAREHOUSE INGEST').collect()
session.sql('USE DATABASE ANALYTICS_DEV_DB').collect()
session.sql('USE SCHEMA ANALYTICS_DEV_DB.DISTRIBUTION').collect()

# REST token for Cortex Analyst
_SF_HOST = f"{_cfg['account']}.snowflakecomputing.com"
_raw_conn = snowflake.connector.connect(**_params)
_SF_TOKEN = _raw_conn.rest.token

print(f'Connected: {session.get_current_account()}')
print(f'Context: {session.get_current_database()}.{session.get_current_schema()}')


In [ ]:
# Demo 1: Live Salesforce opportunities (Zero-Copy — no ETL)
opp_df = session.sql("""
    SELECT opportunity_id, advisor_id, stage,
           ROUND(COALESCE(amount,0)/1e6, 2) AS amount_m,
           close_date,
           row_timestamp AS last_updated
    FROM ANALYTICS_DEV_DB.STAGING.SFDC_OPPORTUNITY
    ORDER BY amount DESC NULLS LAST LIMIT 10
""").to_pandas()
print('LIVE SALESFORCE OPPORTUNITIES (no ETL):')
print(opp_df.to_string(index=False))


In [ ]:
# Demo 2: Distribution signals — SFDC joined with Advisor + Territory dims
signals_df = session.sql("""
    SELECT a.advisor_name, t.territory_name, a.tier,
           COUNT(DISTINCT o.opportunity_id)      AS open_opps,
           ROUND(SUM(COALESCE(o.amount,0))/1e6,2) AS pipeline_m,
           CASE WHEN COUNT(DISTINCT o.opportunity_id) > 3 THEN 'HOT'
                WHEN COUNT(DISTINCT o.opportunity_id) > 1 THEN 'WARM'
                ELSE 'COLD' END AS signal
    FROM ANALYTICS_DEV_DB.STAGING.ADVISOR_DIM a
    JOIN ANALYTICS_DEV_DB.STAGING.TERRITORY_DIM t
      ON a.territory_id = t.territory_id
    LEFT JOIN ANALYTICS_DEV_DB.STAGING.SFDC_OPPORTUNITY o
      ON a.advisor_id = o.advisor_id
     AND o.stage NOT IN ('Closed Won','Closed Lost')
    GROUP BY a.advisor_id, a.advisor_name, t.territory_name, a.tier
    HAVING open_opps > 0
    ORDER BY pipeline_m DESC LIMIT 15
""").to_pandas()
print('DISTRIBUTION SIGNALS (Zero-Copy join):')
print(signals_df.to_string(index=False))


In [ ]:
# Demo 3: Cortex AI brief from the top signals
context = signals_df.head(5).to_dict(orient='records')
prompt = (
    'You are a distribution analytics assistant.\n'
    'Summarize these distribution signals in 3 bullets for a sales manager.\n'
    'Focus on pipeline opportunities, HOT signals, and one action recommendation.\n'
    f'Data: {json.dumps(context, indent=2)}\nSummary:'
)
summary = Complete('mistral-7b', prompt)
print('=== AI DISTRIBUTION BRIEF ===')
print(summary)
